<a href="https://colab.research.google.com/github/ankitta-singh/machinelearning/blob/main/09_Cross_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
from google.colab import files
uploaded = files.upload()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [3]:
import zipfile
with zipfile.ZipFile("house-prices-advanced-regression-techniques.zip", "r") as zip_ref:
    zip_ref.extractall("house data")

In [4]:
df = pd.read_csv("house data/train.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (1460, 81)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePri

In [5]:
x = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

print("Shape of x:", x.shape)
print("Shape of y:", y.shape)

Shape of x: (1460, 80)
Shape of y: (1460,)


In [6]:
numeric_features = x.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = x.select_dtypes(
    include=["object"]
).columns

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 37
Categorical features: 43


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR

In [9]:
models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "SVR": SVR(
        kernel="linear",
        C=1000,
        epsilon=0.5
    )
}

print("Models created:")
for name in models:
    print(name)

Models created:
Linear Regression
Decision Tree
Random Forest
AdaBoost
Gradient Boosting
SVR


In [10]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=200,
    random_state=42
)

print("Tuned Gradient Boosting created successfully!")

Tuned Gradient Boosting created successfully!


In [11]:
from sklearn.pipeline import Pipeline

gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", gb_model)
])

print("Gradient Boosting pipeline created successfully!")

Gradient Boosting pipeline created successfully!


In [12]:
cv_scores_gb = cross_val_score(
    gb_pipeline,
    x,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold CV R2 Scores:")
print(cv_scores_gb)

print("\nMean CV R2:", cv_scores_gb.mean())
print("Std CV R2 :", cv_scores_gb.std())
print("Mean CV R2 %:", cv_scores_gb.mean() * 100)

5-Fold CV R2 Scores:
[0.90366121 0.83707697 0.89702907 0.90224264 0.89507911]

Mean CV R2: 0.8870177995765983
Std CV R2 : 0.025171791918899115
Mean CV R2 %: 88.70177995765984


In [13]:
svr_model_cv = SVR(
    C=1000,
    epsilon=0.5,
    kernel="linear",
    gamma="scale"
)

svr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", svr_model_cv)
])

print("Tuned SVR pipeline created successfully!")

Tuned SVR pipeline created successfully!


In [14]:
cv_scores_svr = cross_val_score(
    svr_pipeline,
    x,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold CV R2 Scores:")
print(cv_scores_svr)

print("\nMean CV R2:", cv_scores_svr.mean())
print("Std CV R2 :", cv_scores_svr.std())
print("Mean CV R2 %:", cv_scores_svr.mean() * 100)

5-Fold CV R2 Scores:
[0.90911308 0.84785813 0.87774725 0.90595745 0.70275491]

Mean CV R2: 0.8486861644245222
Std CV R2 : 0.0762529698977023
Mean CV R2 %: 84.86861644245222


In [15]:
rf_model_cv = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model_cv)
])

print("Random Forest pipeline created successfully!")

Random Forest pipeline created successfully!


In [16]:
rf_model_cv = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    max_features="sqrt",
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model_cv)
])

print("Tuned Random Forest pipeline created successfully!")

Tuned Random Forest pipeline created successfully!


In [17]:
cv_scores_rf = cross_val_score(
    rf_pipeline,
    x,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold CV R2 Scores:")
print(cv_scores_rf)

print("\nMean CV R2:", cv_scores_rf.mean())
print("Std CV R2 :", cv_scores_rf.std())
print("Mean CV R2 %:", cv_scores_rf.mean() * 100)

5-Fold CV R2 Scores:
[0.88560475 0.84118351 0.84029905 0.88998606 0.81181793]

Mean CV R2: 0.8537782600841795
Std CV R2 : 0.02974866723435998
Mean CV R2 %: 85.37782600841794


In [18]:
ada_base = DecisionTreeRegressor(
    max_depth=4,
    random_state=42
)

ada_model_cv = AdaBoostRegressor(
    estimator=ada_base,
    learning_rate=1.0,
    n_estimators=200,
    random_state=42
)

ada_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ada_model_cv)
])

print("Tuned AdaBoost pipeline created successfully!")

Tuned AdaBoost pipeline created successfully!


In [19]:
cv_scores_ada = cross_val_score(
    ada_pipeline,
    x,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold CV R2 Scores:")
print(cv_scores_ada)

print("\nMean CV R2:", cv_scores_ada.mean())
print("Std CV R2 :", cv_scores_ada.std())
print("Mean CV R2 %:", cv_scores_ada.mean() * 100)

5-Fold CV R2 Scores:
[0.84810839 0.8090485  0.84902328 0.85723457 0.78332618]

Mean CV R2: 0.8293481842041285
Std CV R2 : 0.028449276805491898
Mean CV R2 %: 82.93481842041285


In [20]:
from xgboost import XGBRegressor
xgb_model_cv = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    random_state=42,
    n_jobs=-1,
    enable_categorical=False
)

In [21]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model_cv)
])

print("Tuned XGBoost pipeline created successfully!")

Tuned XGBoost pipeline created successfully!


In [22]:
cv_scores_xgb = cross_val_score(
    xgb_pipeline,
    x,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold CV R2 Scores:")
print(cv_scores_xgb)

print("\nMean CV R2:", cv_scores_xgb.mean())
print("Std CV R2 :", cv_scores_xgb.std())
print("Mean CV R2 %:", cv_scores_xgb.mean() * 100)

5-Fold CV R2 Scores:
[0.92063737 0.84636962 0.88756138 0.92537314 0.83603883]

Mean CV R2: 0.8831960678100585
Std CV R2 : 0.03682337409738757
Mean CV R2 %: 88.31960678100586
